In [ ]:
#### Extract the real image from the HPDv3 dataset. ####

import json
import os
from pathlib import Path

all_json_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/HPDv3/all.json"
base_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/HPDv3"

target_real_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/HPDv3/real"
os.makedirs(target_real_image_dir, exist_ok=True)

print(f"Loading JSON file: {all_json_path}")
with open(all_json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Total entries: {len(data)}")

real_image_count = 0
missing_image_count = 0
link_created_count = 0
skipped_count = 0

for idx, entry in enumerate(data):
    path1 = entry.get('path1', '')
    path2 = entry.get('path2', '')
    model1 = entry.get('model1', '')
    model2 = entry.get('model2', '')
    
    real_image_path = None
    if model1 == 'real_images':
        real_image_path = path1
    elif model2 == 'real_images':
        real_image_path = path2
    
    if real_image_path is None:
        skipped_count += 1
        continue
    
    source_path = os.path.join(base_image_dir, real_image_path)
    
    if not os.path.exists(source_path):
        missing_image_count += 1
        if (idx + 1) % 1000 == 0:
            print(f"Processed {idx + 1} entries, missing images: {missing_image_count}")
        continue
    
    filename = os.path.basename(real_image_path)
    
    target_link_path = os.path.join(target_real_image_dir, filename)
    
    if os.path.exists(target_link_path) or os.path.islink(target_link_path):
        real_image_count += 1
        continue
    
    try:
        os.symlink(os.path.abspath(source_path), target_link_path)
        link_created_count += 1
        real_image_count += 1
    except Exception as e:
        print(f"Error creating symlink for {filename}: {e}")
        continue
    
        print(f"Processed {idx + 1}/{len(data)} entries, created {link_created_count} links, real images: {real_image_count}")

print(f"\n完成！")
print(f"总条目数: {len(data)}")
print(f"找到的 real image 数量: {real_image_count}")
print(f"新创建的软链接数量: {link_created_count}")
print(f"缺失的图片数量: {missing_image_count}")
print(f"跳过的条目数量（没有 real_images 模型）: {skipped_count}")
print(f"目标目录: {target_real_image_dir}")

In [ ]:
#### Extract the real image and corresponding prompt from the HPDv3 dataset. ####

import json
import os
from pathlib import Path

all_json_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/HPDv3/all.json"

print(f"Loading JSON file: {all_json_path}")
with open(all_json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Total entries: {len(data)}")

real_image_data = []

for idx, entry in enumerate(data):
    path1 = entry.get('path1', '')
    path2 = entry.get('path2', '')
    model1 = entry.get('model1', '')
    model2 = entry.get('model2', '')
    prompt = entry.get('prompt', '')
    
    real_image_path = None
    if model1 == 'real_images':
        real_image_path = path1
    elif model2 == 'real_images':
        real_image_path = path2
    
    if real_image_path is None:
        continue
    
    filename = os.path.basename(real_image_path)
    uid = os.path.splitext(filename)[0]  # 去掉扩展名，得到 uid
    
    real_image_data.append({
        'uid': uid,
        'prompt': prompt,
        'path': real_image_path
    })
    
    if (idx + 1) % 1000 == 0:
        print(f"Processed {idx + 1}/{len(data)} entries, found {len(real_image_data)} real images")

print(f"\n完成！")
print(f"总条目数: {len(data)}")
print(f"找到的 real_images 数量: {len(real_image_data)}")

# 显示前几个示例
print("\n前 5 个示例：")
for i, item in enumerate(real_image_data[:5]):
    print(f"\n示例 {i+1}:")
    print(f"  UID: {item['uid']}")
    print(f"  Prompt: {item['prompt'][:100]}...")  # 只显示前100个字符

import pandas as pd

output_csv_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/HPDv3/real_images_uid_prompt.csv"
df = pd.DataFrame(real_image_data)

# 检查重复的 uid
uid_counts = df['uid'].value_counts()
duplicate_uids = uid_counts[uid_counts > 1]

print(f"\n=== UID 重复检查 ===")
print(f"总共有 {len(df)} 条记录")
print(f"唯一 UID 数量: {df['uid'].nunique()}")
print(f"重复的 UID 数量: {len(duplicate_uids)}")

if len(duplicate_uids) > 0:
    print(f"\n重复次数最多的前 10 个 UID:")
    for uid, count in duplicate_uids.head(10).items():
        print(f"  UID: {uid}, 出现次数: {count}")
    
    # 显示重复 UID 的详细信息
    print(f"\n重复 UID 的详细信息（前 5 个）:")
    for uid in duplicate_uids.head(5).index:
        duplicate_rows = df[df['uid'] == uid]
        print(f"\n  UID: {uid} (出现 {len(duplicate_rows)} 次)")
        for idx, row in duplicate_rows.iterrows():
            print(f"    - Prompt: {row['prompt'][:80]}...")
            print(f"      Path: {row['path']}")
else:
    print("✓ 没有发现重复的 UID")

# 去重：根据 uid 列去重，保留第一次出现的记录
df_deduplicated = df.drop_duplicates(subset=['uid'], keep='first')
removed_count = len(df) - len(df_deduplicated)

print(f"\n=== 去重处理 ===")
print(f"去重前记录数: {len(df)}")
print(f"去重后记录数: {len(df_deduplicated)}")
print(f"移除的重复记录数: {removed_count}")

# 保存为 CSV 文件（去重后的数据）
df_deduplicated.to_csv(output_csv_path, index=False, encoding='utf-8')
print(f"\n结果已保存到 CSV 文件: {output_csv_path}")
print(f"CSV 文件包含 {len(df_deduplicated)} 行数据（已去重），列名: {list(df_deduplicated.columns)}")
